# Data Fusion: Goodreads, Amazon, DBpedia (Phase III)

**Entity = edition.** Input: the three translated XML files (target schema) and the one-to-one correspondences from Identity Resolution.
Output: **one fused record per real-world edition**, with missing values filled (slot filling) and conflicting values resolved (conflict resolution).

Workflow, following the lecture (WDI06) and the PyDI exercise:
1. Setup, load datasets and correspondences
2. Profile the inputs: density, attribute coverage of the matched records (potential for slot filling)
3. Record groups: group size distribution, suspicious groups caused by matching errors
4. Provenance metadata (trust scores per dataset)
5. Evaluation set for fusion (>= 15 entities, >= 4 attributes, true values from an **external** source)
6. Baseline fusion + consistency, density, accuracy
7. Experiments: conflict resolution functions per attribute, source preference
8. Final strategy, error analysis, summary for the report

Cells marked **TODO (team)** need a decision or manual work from you.

## 1. Setup

In [24]:
import itertools
import logging
import os
import random
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

os.makedirs("logs", exist_ok=True)
# INFO: the fusion engine then prints density and consistency per attribute (needed for the report)
logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s", force=True)

import PyDI.fusion as pf
from PyDI.io import load_xml
from PyDI.utils import DataProfiler
from PyDI.fusion import (DataFusionStrategy, DataFusionEngine, DataFusionEvaluator,
                         longest_string, prefer_higher_trust, union, tokenized_match)
from PyDI.fusion.engine import build_record_groups_from_correspondences

DATA_DIR = Path("data")
CORR_DIR = DATA_DIR / "correspondences"
OUT_DIR = DATA_DIR / "output" / "data_fusion"
EVAL_DIR = Path("gold_standard") / "fusion"          # evaluation set for fusion (NOT the identity-resolution gold standard)
for d in (OUT_DIR, EVAL_DIR):
    d.mkdir(parents=True, exist_ok=True)

NAMES = ["goodreads", "amazon", "dbpedia"]
PAIRS = [("goodreads", "amazon"), ("goodreads", "dbpedia"), ("amazon", "dbpedia")]
SEED = 42

# attributes of the integrated schema that we fuse (>= 5 required)
STRING_ATTRS = ["title", "publisher", "publication_date", "format", "language"]
NUMERIC_ATTRS = ["page_count", "rating", "ratings_count", "price"]
LIST_ATTRS = ["authors_author"]
FUSE_ATTRS = STRING_ATTRS + LIST_ATTRS + NUMERIC_ATTRS
# attributes for which the truth can be looked up externally (rating, ratings_count, price are time-dependent / subjective)
EVAL_ATTRS = ["title", "authors_author", "publisher", "publication_date", "page_count"]
# the evaluation set stores only the YEAR of publication (exact day of an edition is hard to verify externally)
DATE_ATTRS = ["publication_date"]

## 2. Helper functions (pure pandas, no PyDI)

* `is_null`, `to_list`: uniform handling of missing values and list attributes (`authors_author`)
* `tokens`, `similar_values`: the tolerant comparison, same idea as PyDI's `tokenized_match` (lower case, no punctuation, Jaccard on word sets)
* `components`: record groups = connected components of the correspondences (transitive, like the lecture)
* `density`, `consistency`: data-centric evaluation measures (lecture, slides 19-20)
* `attribute_accuracy`: ground-truth-based accuracy per attribute (lecture, slide 21), **strict** and **tolerant**

In [25]:
def is_null(v):
    if isinstance(v, (list, tuple, set)):
        return len(v) == 0
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except (TypeError, ValueError):
        pass
    return str(v).strip() in ("", "nan", "<NA>", "None")


def to_list(v):
    """authors_author -> list of non-empty strings, or None."""
    if isinstance(v, (list, tuple, set)):
        out = [str(x).strip() for x in v if not is_null(x)]
    elif is_null(v):
        out = []
    else:
        out = [str(v).strip()]
    return out or None


def tokens(v):
    """Set of lower-cased alphanumeric words; lists are flattened (all words of all names)."""
    if isinstance(v, (list, tuple, set)):
        v = " ".join(str(x) for x in v)
    s = unicodedata.normalize("NFKD", str(v)).encode("ascii", "ignore").decode().lower()
    return set(re.sub(r"[^a-z0-9 ]", " ", s).split())


def tok_jaccard(a, b):
    ta, tb = tokens(a), tokens(b)
    return len(ta & tb) / len(ta | tb) if (ta | tb) else 1.0


def strict_equal(a, b):
    """Exact comparison after trimming / lower-casing; lists compare as sets; numbers numerically."""
    if isinstance(a, (list, tuple)) or isinstance(b, (list, tuple)):
        return {str(x).strip().lower() for x in (a or [])} == {str(x).strip().lower() for x in (b or [])}
    try:
        return float(a) == float(b)
    except (TypeError, ValueError):
        return str(a).strip().lower() == str(b).strip().lower()


# Titles: the sources append disambiguation / series / genre notes to the title ("Ilium (novel)", "Crown of Midnight (Throne of Glass)",
# "Dear Wife: A Novel"). These are removed from the END of the title before fusion (applied in load(); the original stays in title_raw).
TRAIL_PAREN = re.compile(r"\s*\([^()]*\)\s*$")
TRAIL_GENRE = re.compile(r"\s*:\s*an? (novel|thriller|memoir|mystery|romance|novella)\b.*$", re.I)


def clean_title(v):
    if not isinstance(v, str):
        return v
    s = v.strip()
    while True:                                   # repeat: "X: A Novel (Series, 2)" -> "X: A Novel" -> "X"
        t = TRAIL_GENRE.sub("", TRAIL_PAREN.sub("", s)).strip()
        if t == s or not t:
            break
        s = t
    return s


# Author lists: every source annotates names differently, so the cleaning is source-specific (applied in load(); original in authors_raw).
# * goodreads: parentheses are ROLES. "(Goodreads Author)", "(Pseudonym)", ... mark authors; "(Translator)", "(Illustrator)",
#   "(Editor)", "(Introduction)", ... mark contributors who are not authors -> dropped (kept only if a book has no author at all,
#   e.g. an anthology with editors only). Broken fragments like "Introduction and Notes)" -> dropped
# * dbpedia: parentheses are DISAMBIGUATION ("Paul Cornell (writer)") -> removed, the name is kept; entries that are works
#   ("Love and War (Cornell novel)") -> dropped; joined names ("James Patterson and David Ellis") -> split
# * all: split "A and B" only if both parts have >= 2 words (keeps "Dick and Angel Strawbridge"), remove duplicates
AUTHOR_ROLES = {"goodreads author", "author", "writer", "pseudonym", "creator"}
PAREN = re.compile(r"\s*\(([^()]*)\)\s*")
JOINED = re.compile(r"\s+(?:and|&)\s+")


def split_joined(name):
    parts = JOINED.split(name)
    return parts if len(parts) > 1 and all(len(p.split()) >= 2 for p in parts) else [name]


def is_author_role(note):
    return note in AUTHOR_ROLES or "author" in note           # also "co-author", "author & illustrator", "author/illustrator"


def clean_authors(names, source):
    if not isinstance(names, list):
        return names
    out, contributors = [], []
    for x in names:
        x = re.sub(r"\s+", " ", str(x)).strip()
        if not x or (x.endswith(")") and "(" not in x):
            continue
        notes = [m.lower().strip() for m in re.findall(r"\(([^()]*)\)", x)]
        if source == "goodreads" and not all(is_author_role(n) for n in notes):
            contributors.append(PAREN.sub(" ", x).strip())   # kept only as fallback (e.g. anthologies with editors only)
            continue
        if any(re.search(r"\b(novel|novella|book)\b", n) for n in notes):
            continue
        out.extend(split_joined(PAREN.sub(" ", x).strip()))
    if not out:                                               # no author at all -> better the contributors than nothing
        out = contributors
    seen, res = set(), []
    for x in out:
        if x.lower() not in seen:
            seen.add(x.lower())
            res.append(x)
    return res or None


# Page counts differ slightly between printings / counting conventions (front matter, index): 470 vs 471, 545 vs 548.
# The tolerant comparison accepts a relative difference of up to 2 %; the strict comparison stays exact.
NUMERIC_TOL = {"page_count": 0.02}


def relative_tolerance_match(fused_value, expected_value, tolerance=0.02):
    """Evaluation function (PyDI signature): |fused - expected| <= tolerance * expected."""
    if is_null(fused_value) or is_null(expected_value):
        return is_null(fused_value) and is_null(expected_value)
    try:
        f, e = float(fused_value), float(expected_value)
    except (TypeError, ValueError):
        return False
    return abs(f - e) <= tolerance * abs(e)


# Publisher names: the same publisher is written in many ways ("Riverhead" / "Riverhead Books", "Hill and Wang" / "Hill & Wang",
# "G. P. Putnam's Sons" / "G.P. Putnam's Sons"). Legal-form and generic words are removed before comparing.
# Imprints stay different ("HarperTorch" vs "Harper"). Known false positive: "Penguin Books" == "Penguin Press".
PUB_STOP = {"books", "book", "publishing", "publishers", "publisher", "publications", "publication", "press", "pr",
            "inc", "ltd", "llc", "limited", "co", "company", "corp", "corporation", "group", "pvt", "verlag",
            "uk", "usa", "us", "the", "and", "sons"}


def pub_tokens(v):
    """Normalised word set of a publisher name (ASCII, lower case, '&' = 'and', initials joined, generic words removed)."""
    s = unicodedata.normalize("NFKD", str(v)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"\(.*?\)", " ", s).replace("&", " and ").replace("'", "").replace(".", " ")
    s = re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()
    s = re.sub(r"\b([a-z])\s(?=[a-z]\b)", r"\1", s)             # "g p putnams" -> "gp putnams"
    toks = set(s.split())
    return (toks - PUB_STOP) or toks


def publisher_match(fused_value, expected_value, threshold=0.8):
    """Evaluation function (PyDI signature): Jaccard of the normalised publisher words >= threshold."""
    if is_null(fused_value) or is_null(expected_value):
        return is_null(fused_value) and is_null(expected_value)
    ta, tb = pub_tokens(fused_value), pub_tokens(expected_value)
    return len(ta & tb) / len(ta | tb) >= threshold


def year_of(v):
    m = re.search(r"\d{4}", str(v))
    return m.group(0) if m else None


def similar_values(a, b, attr, thr=0.8):
    """Tolerant comparison: word-Jaccard >= thr for text/lists (normalised names for publisher), exact for numbers (page_count +-2 %), same year for dates."""
    if attr in NUMERIC_ATTRS:
        if attr in NUMERIC_TOL:
            return relative_tolerance_match(a, b, NUMERIC_TOL[attr])
        return strict_equal(a, b)
    if attr == "publisher":
        return publisher_match(a, b, thr)
    if attr in DATE_ATTRS:
        return year_of(a) is not None and year_of(a) == year_of(b)
    return tok_jaccard(a, b) >= thr


def components(*corr_dfs):
    """Union-find over all correspondences. Returns {record id: group id}."""
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for c in corr_dfs:
        for a, b in zip(c["id1"], c["id2"]):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[rb] = ra
    return {x: find(x) for x in list(parent)}


def group_table(corr_dfs, source_of):
    """One row per record group: size and number of records per dataset."""
    comp = components(*corr_dfs)
    members = defaultdict(list)
    for rid, g in comp.items():
        members[g].append(rid)
    rows = []
    for g, ids in members.items():
        cnt = Counter(source_of[i] for i in ids)
        rows.append({"group": g, "size": len(ids), **{n: cnt.get(n, 0) for n in NAMES}, "ids": sorted(ids)})
    t = pd.DataFrame(rows)
    t["clean"] = (t["size"] <= len(NAMES)) & (t[NAMES].max(axis=1) <= 1)   # at most one record per dataset
    return t


def drop_dirty_correspondences(corr_dfs, gt):
    """Remove all correspondences that lie inside a group that is not clean."""
    comp = components(*corr_dfs)
    dirty = set(gt.loc[~gt["clean"], "group"])
    return [c[[comp[a] not in dirty for a in c["id1"]]].reset_index(drop=True) for c in corr_dfs]


def density(df, attrs):
    """Fraction of non-null values per attribute and for the whole table (lecture, slide 19)."""
    col = {a: float((~df[a].map(is_null)).mean()) for a in attrs if a in df.columns}
    col["TABLE"] = float(np.mean([col[a] for a in col]))
    return pd.Series(col)


def consistency(datasets, gt, attrs, thr=None):
    """Share of groups (>= 2 records) without conflicting values per attribute (lecture, slide 20).
    thr=None: values must be identical (after trimming/lower case); thr=0.8: similar values count as equal."""
    by_id = {}
    for df in datasets:
        d = df.set_index("id")
        for a in attrs:
            by_id.setdefault(a, {}).update(d[a].to_dict())
    out = {}
    groups = gt[gt["size"] >= 2]
    for a in attrs:
        ok = 0
        for ids in groups["ids"]:
            vals = [by_id[a][i] for i in ids if i in by_id[a] and not is_null(by_id[a][i])]
            if thr is None:
                good = all(strict_equal(vals[0], v) for v in vals[1:])
            else:
                good = all(similar_values(vals[0], v, a, thr) for v in vals[1:])
            ok += good
        out[a] = ok / max(len(groups), 1)
    s = pd.Series(out)
    s["TABLE"] = s.mean()
    return s


def attach_eval_id(fused, expected_ids):
    """The evaluation set names each entity by the ID of ONE of its records; _fusion_sources lists all fused IDs."""
    out = fused.copy()
    out["eval_id"] = out["_fusion_sources"].apply(lambda s: next((x for x in s if x in expected_ids), None))
    return out


def attribute_accuracy(fused, expected, attrs, thr=0.8):
    """Accuracy per attribute against the evaluation set: strict and tolerant (see similar_values)."""
    f = fused[fused["eval_id"].notna()].drop_duplicates("eval_id").set_index("eval_id")
    ex = expected.set_index("id")
    rows = []
    for a in attrs:
        n = s = t = 0
        for eid, true in ex[a].items() if a in ex.columns else []:
            if is_null(true) or eid not in f.index:
                continue
            got = f.at[eid, a]
            n += 1
            # the evaluation set stores only the year of publication_date -> dates are compared by year (strict and tolerant)
            s += (not is_null(got)) and (similar_values(got, true, a) if a in DATE_ATTRS else strict_equal(got, true))
            t += (not is_null(got)) and similar_values(got, true, a, thr)
        rows.append({"attribute": a, "values": n, "accuracy strict": s / n if n else np.nan,
                     "accuracy tolerant": t / n if n else np.nan})
    r = pd.DataFrame(rows).set_index("attribute")
    r.loc["OVERALL"] = [r["values"].sum(),
                        (r["accuracy strict"] * r["values"]).sum() / max(r["values"].sum(), 1),
                        (r["accuracy tolerant"] * r["values"]).sum() / max(r["values"].sum(), 1)]
    return r

## 3. Load datasets and correspondences

In [26]:
def load(name):
    df = load_xml(DATA_DIR / f"{name}.xml", name=name)
    df.attrs["source"] = name
    for c in FUSE_ATTRS:
        if c not in df.columns:
            df[c] = None
    for c in NUMERIC_ATTRS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["authors_author"] = df["authors_author"].map(to_list)
    df["authors_raw"] = df["authors_author"]
    df["authors_author"] = df["authors_author"].map(lambda v: clean_authors(v, name))
    df["title_raw"] = df["title"]
    df["title"] = df["title"].map(clean_title)
    return df


dfs = {n: load(n) for n in NAMES}
source_of = {i: n for n, df in dfs.items() for i in df["id"]}
assert len(source_of) == sum(len(d) for d in dfs.values()), "record ids must be unique across the datasets"


def load_corr(a, b):
    c = pd.read_csv(CORR_DIR / f"{a}_{b}.csv")
    if "similarity" not in c.columns:                 # the exercise files use the column name 'similarity'
        c["similarity"] = c["score"]
    return c[["id1", "id2", "similarity"]]


corr = {f"{a}_{b}": load_corr(a, b) for a, b in PAIRS}
for n, df in dfs.items():
    print(f"{n:<10} {len(df):>7,} records")
for n, df in dfs.items():
    changed = (df["title"] != df["title_raw"].str.strip()) & df["title_raw"].notna()
    print(f"{n:<10} title cleaned for {changed.sum():>6,} records ({changed.mean():.1%})")
    changed = df["authors_author"].astype(str) != df["authors_raw"].astype(str)
    print(f"{n:<10} author list cleaned for {changed.sum():>6,} records ({changed.mean():.1%})")
for p, c in corr.items():
    print(f"{p:<20} {len(c):>7,} correspondences")

goodreads   52,478 records
amazon      40,000 records
dbpedia     59,270 records
goodreads  title cleaned for  1,041 records (2.0%)
goodreads  author list cleaned for 29,103 records (55.5%)
amazon     title cleaned for 17,985 records (45.0%)
amazon     author list cleaned for     20 records (0.1%)
dbpedia    title cleaned for 12,370 records (20.9%)
dbpedia    author list cleaned for  5,452 records (9.2%)
goodreads_amazon       3,703 correspondences
goodreads_dbpedia      7,639 correspondences
amazon_dbpedia         3,558 correspondences


## 4. Profile the inputs

**Density** of the input datasets and the **potential for slot filling**: only records that have a correspondence can fill gaps in other datasets,
so the coverage table is computed on the matched records only (like Task 1 of the exercise).

In [27]:
dens = pd.DataFrame({n: density(df, FUSE_ATTRS) for n, df in dfs.items()})
print("=== Density of the input datasets (all records) ===")
display(dens.round(3))

matched_ids = set(pd.concat([c[["id1", "id2"]].stack() for c in corr.values()]))
print("\n=== Records with a correspondence, per dataset ===")
display(pd.Series({n: int(df["id"].isin(matched_ids).sum()) for n, df in dfs.items()}))

matched_dens = pd.DataFrame({n: density(df[df["id"].isin(matched_ids)], FUSE_ATTRS) for n, df in dfs.items()})
print("=== Density of the matched records only (which dataset can fill which attribute?) ===")
display(matched_dens.round(3))

=== Density of the input datasets (all records) ===


,goodreads,amazon,dbpedia
title,1.000,1.000,1.000
publisher,0.930,0.953,0.795
publication_date,0.904,0.951,0.802
format,0.972,0.916,0.000
language,0.927,0.990,0.796
authors_author,1.000,1.000,0.909
page_count,0.955,0.705,0.616
rating,1.000,1.000,0.000
ratings_count,1.000,1.000,0.000
price,0.726,0.882,0.000



=== Records with a correspondence, per dataset ===


goodreads    10452
amazon        6573
dbpedia      10110
dtype: int64

=== Density of the matched records only (which dataset can fill which attribute?) ===


,goodreads,amazon,dbpedia
title,1.000,1.000,1.000
publisher,0.969,0.913,0.772
publication_date,0.923,0.913,0.845
format,0.988,0.982,0.000
language,0.955,0.985,0.807
authors_author,1.000,1.000,0.955
page_count,0.981,0.868,0.750
rating,1.000,1.000,0.000
ratings_count,1.000,1.000,0.000
price,0.844,0.887,0.000


## 5. Record groups and group size distribution

All correspondences of the three pairs are chained: records that (transitively) describe the same edition end up in one group.
Because every source lists an edition at most once and each pair is matched one-to-one, a **valid group has at most 3 records, one per dataset**.
Larger groups, or groups with two records of the same dataset, can only come from wrong correspondences that got chained together.

In [28]:
corr_list = [corr[f"{a}_{b}"] for a, b in PAIRS]

# PyDI's own grouping (prints the group size distribution, as in the exercise)
record_groups = build_record_groups_from_correspondences(list(dfs.values()), corr_list, id_column="id")

gt = group_table(corr_list, source_of)
print("\n=== Group size distribution (own computation, same grouping) ===")
dist = gt["size"].value_counts().sort_index().rename("groups").to_frame()
dist["share"] = (dist["groups"] / dist["groups"].sum()).round(4)
display(dist)
print(f"groups: {len(gt):,} | clean groups (<= 1 record per dataset): {gt['clean'].sum():,} | "
      f"not clean: {(~gt['clean']).sum():,} ({(~gt['clean']).mean():.1%})")
print("\n=== Composition of the groups (records per dataset) ===")
display(gt.groupby(NAMES).size().rename("groups").reset_index().sort_values("groups", ascending=False).head(12))

[INFO] Created 137313 record groups from 14900 correspondences
[INFO] Group Size Distribution of 137313 clusters:
[INFO] 	Cluster Size	| Frequency	| Percentage
[INFO] 	──────────────────────────────────────────────────
[INFO] 		2	|	10965	|	7.99%
[INFO] 		3	|	1735	|	1.26%



=== Group size distribution (own computation, same grouping) ===


,groups,share
size,,
2,10965,0.8634
3,1735,0.1366


groups: 12,700 | clean groups (<= 1 record per dataset): 12,700 | not clean: 0 (0.0%)

=== Composition of the groups (records per dataset) ===


,goodreads,amazon,dbpedia,groups
1,1,0,1,6127
2,1,1,0,2590
0,0,1,1,2248
3,1,1,1,1735


In [29]:
# Look at a few suspicious groups: do the records really describe the same edition?
COLS = ["title", "authors_author", "publisher", "publication_date", "page_count", "format"]
by_id = pd.concat([df.set_index("id")[COLS].assign(dataset=n) for n, df in dfs.items()])
pd.set_option("display.max_colwidth", 45)
for _, g in gt[~gt["clean"]].sort_values("size", ascending=False).head(3).iterrows():
    print(f"\n--- group of size {g['size']} ---")
    display(by_id.loc[g["ids"]])

### Decision: keep or remove suspicious groups?

`FILTER_MODE = "none"` fuses every group as it is. `"clean_only"` removes all correspondences inside groups that are not clean; their records stay in the
integrated dataset as unfused singletons. Run both, compare density / consistency / accuracy and write down which one you use and why
(trade-off: fewer wrong merges vs. fewer entities with filled values).

In [30]:
FILTER_MODE = "none"        # "none" or "clean_only"

if FILTER_MODE == "clean_only":
    corr_list_used = drop_dirty_correspondences(corr_list, gt)
    print("correspondences:", [len(c) for c in corr_list], "->", [len(c) for c in corr_list_used])
else:
    corr_list_used = corr_list
gt_used = group_table(corr_list_used, source_of)
print(f"groups used for fusion: {len(gt_used):,} (size distribution: {gt_used['size'].value_counts().sort_index().to_dict()})")

groups used for fusion: 12,700 (size distribution: {2: 10965, 3: 1735})


## 6. Provenance metadata: trust per dataset

`prefer_higher_trust` reads the score from `df.attrs["trust"]`. We start with **equal trust** (no prior knowledge; ties go to the dataset that comes first in the list)
and let the evaluation set tell us which source is reliable for which attribute (section 9), as in the DBpedia case study of the lecture, where the best
function was learned on a ground truth.
Also document the **dataset date** (when the data was collected). **TODO (team):** fill in the real collection dates / your own quality score and explain in the report why you set them this way.

In [31]:
PRIOR_TRUST = {"goodreads": 1.0, "amazon": 1.0, "dbpedia": 1.0}
DATASET_DATE = {"goodreads": 2020, "amazon": 2023, "dbpedia": 2026}


def set_trust(trust):
    for n, df in dfs.items():
        df.attrs["trust"] = float(trust[n])
        df.attrs["dataset_date"] = DATASET_DATE[n]


def trust_from_order(order):
    """['amazon','goodreads','dbpedia'] -> amazon 3, goodreads 2, dbpedia 1"""
    return {n: len(order) - i for i, n in enumerate(order)}


set_trust(PRIOR_TRUST)

## 7. Evaluation set for fusion

Requirements (exercise slides): **>= 15 entities (more is better), >= 4 attributes per entity**, true values looked up manually in an **external** source
(not one of our three datasets, e.g. Open Library, Library of Congress, WorldCat, publisher page; ISBNs below make the lookup unambiguous).
It should contain entities that **actually have conflicts**. Identify each entity by the ID of **one** of its records (never two records of the same group).

Because the entities are edition-level, look up the **edition** (ISBN), not the work.

**Step 1** exports candidate groups (clean groups with conflicts) to a CSV. **Step 2** (manual): fill the `TRUE__*` columns, separate several authors by `;`.
**Step 3** turns the filled CSV into the XML evaluation set in the target schema.

In [32]:
N_CANDIDATES = 60          # export more than you need; skip entities you cannot verify
CAND_PATH = EVAL_DIR / "fusion_eval_candidates.csv"
EVAL_XML = EVAL_DIR / "fusion_evaluation_set.xml"

# optional: ISBNs from the raw files (helps to look up the right edition); needs the raw files from the schema-mapping phase
isbn_lookup = {}
try:
    from build_gold_standard import raw_isbns, xml_titles, isbn13s
    raw = {"goodreads": ("data/books_1.Best_Books_Ever.csv", "goodreads_id_"),
           "amazon": ("data/amazon_books_subset.csv", "amazon_id_"),
           "dbpedia": ("data/dbpedia_book_data.xml", "dbpedia_id_")}
    for n, (path, prefix) in raw.items():
        isbn_lookup.update({k: ", ".join(sorted(isbn13s(v))) for k, v in raw_isbns(n, path, xml_titles(DATA_DIR / f"{n}.xml"), prefix).items()})
except Exception as e:                                      # noqa
    print("No ISBNs available:", repr(e))


def conflicting_attrs(ids):
    n = 0
    for a in EVAL_ATTRS:
        vals = [by_attr[a][i] for i in ids if not is_null(by_attr[a].get(i))]
        n += len(vals) >= 2 and not all(similar_values(vals[0], v, a, 0.999) for v in vals[1:])
    return n


by_attr = {a: {} for a in EVAL_ATTRS}
for df in dfs.values():
    for a in EVAL_ATTRS:
        by_attr[a].update(df.set_index("id")[a].to_dict())

if not CAND_PATH.exists():
    clean = gt_used[(gt_used["clean"]) & (gt_used["size"] >= 2)].copy()
    clean["n_conflicts"] = clean["ids"].map(conflicting_attrs)
    pool = clean[clean["n_conflicts"] >= 2]
    sample = pool.sample(min(N_CANDIDATES, len(pool)), random_state=SEED)
    rows = []
    for _, g in sample.iterrows():
        r = {"eval_id": g["ids"][0], "members": " | ".join(g["ids"]),
             "isbn(s)": " / ".join(filter(None, (isbn_lookup.get(i, "") for i in g["ids"])))}
        for i in g["ids"]:
            for a in EVAL_ATTRS:
                v = by_attr[a].get(i)
                r[f"{a}__{source_of[i]}"] = "; ".join(v) if isinstance(v, list) else v
        r.update({f"TRUE__{a}": "" for a in EVAL_ATTRS})
        rows.append(r)
    pd.DataFrame(rows).to_csv(CAND_PATH, index=False)
    print(f"{len(rows)} candidates written to {CAND_PATH}  ->  fill the TRUE__ columns manually")
else:
    print("candidate file exists:", CAND_PATH)

  goodreads: ISBNs taken from data/books_1.Best_Books_Ever.csv by id; titles line up for 99.99%
  amazon: ISBNs taken from data/amazon_books_subset.csv by id; titles line up for 100.00%
  dbpedia: ISBNs taken from data/dbpedia_book_data.xml by id; titles line up for 100.00%
candidate file exists: gold_standard/fusion/fusion_eval_candidates.csv


In [33]:
def build_eval_xml(filled_csv, xml_path):
    """Filled candidate CSV -> XML evaluation set in the target schema (same structure as our dataset XMLs)."""
    import xml.etree.ElementTree as ET
    d = pd.read_csv(filled_csv, dtype=str)
    root, kept = ET.Element("books"), 0
    for _, r in d.iterrows():
        truth = {a: r.get(f"TRUE__{a}") for a in EVAL_ATTRS if not is_null(r.get(f"TRUE__{a}"))}
        if len(truth) < 4:
            continue                                         # requirement: >= 4 attributes per entity
        b = ET.SubElement(root, "book", id=r["eval_id"])
        for a, v in truth.items():
            if a == "authors_author":
                au = ET.SubElement(b, "authors")
                for name in str(v).split(";"):
                    if name.strip():
                        ET.SubElement(au, "author").text = name.strip()
            else:
                ET.SubElement(b, a).text = str(v).strip()
        kept += 1
    ET.ElementTree(root).write(xml_path, encoding="utf-8", xml_declaration=True)
    print(f"{kept} entities with >= 4 true attributes written to {xml_path}")


FILLED_CSV = EVAL_DIR / "fusion_eval_filled.csv"           # TODO (team): copy of the candidate file with the TRUE__ columns filled
if FILLED_CSV.exists():
    build_eval_xml(FILLED_CSV, EVAL_XML)

29 entities with >= 4 true attributes written to gold_standard/fusion/fusion_evaluation_set.xml


In [34]:
# Creating a second candidate set for fusion evaluation (to be filled manually) to get more entries for the (overall) evaluation set

N_MORE = 30
CAND2_PATH = EVAL_DIR / "fusion_eval_candidates_2.csv"
ONLY_ISBN_CONSISTENT = True


def isbn_set(i):
    return set(isbn_lookup.get(i, "").split(", ")) - {""}


def isbns_agree(ids):
    sets = [isbn_set(i) for i in ids if isbn_set(i)]
    return len(sets) >= 2 and bool(set.intersection(*sets))


old_ids = set(pd.read_csv(CAND_PATH)["eval_id"])
clean = gt_used[(gt_used["clean"]) & (gt_used["size"] >= 2)].copy()
clean["n_conflicts"] = clean["ids"].map(conflicting_attrs)
pool = clean[(clean["n_conflicts"] >= 2) & ~clean["ids"].map(lambda ids: ids[0] in old_ids)]
if ONLY_ISBN_CONSISTENT:
    pool = pool[pool["ids"].map(isbns_agree)]
print("available new candidates:", len(pool), "| ISBNs loaded for", len(isbn_lookup), "records")

sample = pool.sample(min(N_MORE, len(pool)), random_state=SEED + 1)
rows = []
for _, g in sample.iterrows():
    by_src = {source_of[i]: i for i in g["ids"]}          # clean groups: at most one record per dataset
    sets = [isbn_set(i) for i in g["ids"] if isbn_set(i)]
    r = {"eval_id": g["ids"][0],
         "isbn_common": ", ".join(sorted(set.intersection(*sets))) if len(sets) >= 2 else ""}
    for n in NAMES:
        r[f"isbn__{n}"] = isbn_lookup.get(by_src[n], "") if n in by_src else ""
    for a in EVAL_ATTRS:
        for n in NAMES:
            v = by_attr[a].get(by_src[n]) if n in by_src else None
            r[f"{a}__{n}"] = "; ".join(v) if isinstance(v, list) else v
        r[f"TRUE__{a}"] = ""
    r["members"] = " | ".join(g["ids"])
    rows.append(r)
pd.DataFrame(rows).to_csv(CAND2_PATH, index=False)
print(len(rows), "new candidates ->", CAND2_PATH)

available new candidates: 1112 | ISBNs loaded for 151748 records
30 new candidates -> gold_standard/fusion/fusion_eval_candidates_2.csv


In [35]:
# Combine the two filled evaluation files (fusion_eval_filled.csv and fusion_eval_filled_2.csv) into one file (fusion_eval_filled_all.csv) and build the evaluation XML from it.

a = pd.read_csv(EVAL_DIR / "fusion_eval_filled.csv", dtype=str)
b = pd.read_csv(EVAL_DIR / "fusion_eval_filled_2.csv", dtype=str)
all_filled = pd.concat([a, b], ignore_index=True).drop_duplicates("eval_id")
all_filled.to_csv(EVAL_DIR / "fusion_eval_filled_all.csv", index=False)
build_eval_xml(EVAL_DIR / "fusion_eval_filled_all.csv", EVAL_XML)

59 entities with >= 4 true attributes written to gold_standard/fusion/fusion_evaluation_set.xml


In [36]:
# Load the evaluation set (if it exists) and split it: dev part to choose functions, test part for the final number
df_eval = None
if EVAL_XML.exists():
    df_eval = load_xml(EVAL_XML, name="fusion_gold")
    for c in NUMERIC_ATTRS:
        if c in df_eval.columns:
            df_eval[c] = pd.to_numeric(df_eval[c], errors="coerce")
    if "authors_author" in df_eval.columns:
        df_eval["authors_author"] = df_eval["authors_author"].map(to_list)
    assert set(df_eval["id"]) <= set(source_of), "evaluation IDs must be record IDs of our datasets"
    ids_sorted = sorted(df_eval["id"])
    random.Random(SEED).shuffle(ids_sorted)
    if len(ids_sorted) >= 30:
        half = len(ids_sorted) // 2
        eval_dev_ids, eval_test_ids = set(ids_sorted[:half]), set(ids_sorted[half:])
    else:
        print("WARNING: fewer than 30 entities -> dev and test are identical (results will be optimistic)")
        eval_dev_ids = eval_test_ids = set(ids_sorted)
    df_eval_dev = df_eval[df_eval["id"].isin(eval_dev_ids)].reset_index(drop=True)
    df_eval_test = df_eval[df_eval["id"].isin(eval_test_ids)].reset_index(drop=True)
    expected_ids = set(df_eval["id"])
    print(f"evaluation set: {len(df_eval)} entities (dev {len(df_eval_dev)}, test {len(df_eval_test)}); attributes: "
          f"{[c for c in df_eval.columns if c in FUSE_ATTRS]}")
else:
    expected_ids = set()
    print("No evaluation set yet -> sections that need it are skipped. Create it first (steps 1-3 above).")

evaluation set: 59 entities (dev 29, test 30); attributes: ['title', 'authors_author', 'publisher', 'publication_date', 'page_count']


## 8. Baseline fusion

Simple functions per attribute (lecture slide 12): longest string for text, union for lists, median for numbers.
The engine prints **density** and **consistency**; copy these numbers into the report. A low consistency means many conflicts, so fusion matters for that attribute.

**Own conflict resolution function:** `publisher_vote` (defined below) votes on *normalised* publisher names, so spelling variants of one
publisher support each other. The tolerant comparison of publisher uses the same normalisation (`publisher_match`); the strict comparison stays exact.

In [37]:
def publisher_vote(values, sources=None, source_datasets=None, trust_map=None, **kwargs):
    """OWN conflict resolution function for publisher.
    1. group the values by their normalised name (pub_tokens), so spelling variants of one publisher vote together
       ("Viking" + "Viking Press" = 2 votes against "Penguin" = 1 vote; plain voting sees three different strings)
    2. the group with the most votes wins; ties -> the group with the most trusted source (per-attribute trust_map,
       chosen in the grid like for prefer_higher_trust), else the first value
    3. representative of the winning group: most frequent spelling, then the longest (= most complete name)
    Returns (value, confidence, metadata) like the PyDI functions."""
    sources = sources or [None] * len(values)
    pairs = [(v, (source_datasets or {}).get(s)) for v, s in zip(values, sources) if not is_null(v)]
    if not pairs:
        return None, 0.0, {"reason": "no_valid_values"}
    clusters = {}
    for v, ds in pairs:
        clusters.setdefault(" ".join(sorted(pub_tokens(v))), []).append((v, ds))
    trust = trust_map or {}
    key, members = max(clusters.items(), key=lambda kv: (len(kv[1]), max(trust.get(ds, 0) for _, ds in kv[1])))
    spellings = Counter(v for v, _ in members)
    value = max(spellings, key=lambda v: (spellings[v], len(str(v))))
    return value, len(members) / len(pairs), {"normalized_key": key, "clusters": len(clusters), "votes": len(members)}


CUSTOM_FUNCTIONS = {"publisher_vote": publisher_vote}     # own functions, found by fn() like the PyDI ones


def fn(*names):
    """First conflict resolution function (own ones first, then PyDI.fusion) that exists under one of the given names."""
    for n in names:
        f = CUSTOM_FUNCTIONS.get(n) or getattr(pf, n, None)
        if f is not None:
            return f
    return None


print("available in PyDI.fusion:", sorted(n for n in dir(pf) if not n.startswith("_") and callable(getattr(pf, n))))


def add_eval_functions(strategy):
    """Same tolerant comparison in PyDI's evaluator as in our own attribute_accuracy."""
    for a in ["title", "authors_author", "format"]:
        strategy.add_evaluation_function(a, tokenized_match, threshold=0.8)
    strategy.add_evaluation_function("publisher", publisher_match, threshold=0.8)    # normalised publisher names
    strategy.add_evaluation_function("publication_date", pf.year_only_match)   # evaluation set has years only
    strategy.add_evaluation_function("page_count", relative_tolerance_match, tolerance=NUMERIC_TOL["page_count"])

available in PyDI.fusion: ['AttributeCoverageAnalyzer', 'AttributeValueFuser', 'ConflictResolutionFunction', 'DataFusionEngine', 'DataFusionEvaluator', 'DataFusionStrategy', 'FusionContext', 'FusionReport', 'FusionResult', 'ProvenanceInfo', 'ProvenanceTracker', 'RecordGroup', 'analyze_attribute_coverage', 'analyze_conflicts_preview', 'average', 'boolean_match', 'build_record_groups_from_correspondences', 'calculate_consistency_metrics', 'calculate_coverage_metrics', 'compare_dataset_schemas', 'create_empty_strategy', 'detect_attribute_conflicts', 'earliest', 'exact_match', 'favour_sources', 'intersection', 'intersection_k_sources', 'longest_string', 'maximum', 'median', 'minimum', 'most_complete', 'most_recent', 'numeric_tolerance_match', 'prefer_higher_trust', 'print_conflict_preview', 'random_value', 'set_equality_match', 'shortest_string', 'sum_values', 'tokenized_match', 'union', 'voting', 'weighted_voting', 'year_only_match']


In [38]:
median_fn = fn("median", "median_value")
BASELINE = {"title": longest_string, "publisher": longest_string, "publication_date": longest_string,
            "format": longest_string, "language": longest_string, "authors_author": union,
            "page_count": median_fn or prefer_higher_trust, "rating": median_fn or prefer_higher_trust,
            "ratings_count": median_fn or prefer_higher_trust, "price": median_fn or prefer_higher_trust}

set_trust(PRIOR_TRUST)
baseline_strategy = DataFusionStrategy("Version_1_Baseline")
for a, f in BASELINE.items():
    baseline_strategy.add_attribute_fuser(a, f)
add_eval_functions(baseline_strategy)

baseline_engine = DataFusionEngine(baseline_strategy, debug=True, debug_file=str(OUT_DIR / "fusion_debug_baseline.jsonl"))
fused_baseline = baseline_engine.run(datasets=list(dfs.values()), correspondences=corr_list_used,
                                     id_column="id", include_singletons=True)
print(len(fused_baseline), "fused records (including singletons)")
fused_baseline.head(3)

[INFO] Registered fuser for attribute 'title' using rule 'longest_string'
[INFO] Registered fuser for attribute 'publisher' using rule 'longest_string'
[INFO] Registered fuser for attribute 'publication_date' using rule 'longest_string'
[INFO] Registered fuser for attribute 'format' using rule 'longest_string'
[INFO] Registered fuser for attribute 'language' using rule 'longest_string'
[INFO] Registered fuser for attribute 'authors_author' using rule 'union'
[INFO] Registered fuser for attribute 'page_count' using rule 'median'
[INFO] Registered fuser for attribute 'rating' using rule 'median'
[INFO] Registered fuser for attribute 'ratings_count' using rule 'median'
[INFO] Registered fuser for attribute 'price' using rule 'median'
[INFO] Registered evaluation function for attribute 'title' with params {'threshold': 0.8}
[INFO] Registered evaluation function for attribute 'authors_author' with params {'threshold': 0.8}
[INFO] Registered evaluation function for attribute 'format' with pa

137313 fused records (including singletons)


,_id,_fusion_sources,_fusion_source_datasets,authors_author,authors_raw,awards_award,edition,format,genres_genre,id,...,price,publication_date,publisher,rating,ratings_count,series,title,title_raw,_fusion_confidence,_fusion_metadata
0,amazon_id_947,"[goodreads_id_1, amazon_id_947]","[goodreads, amazon]",[Suzanne Collins],[Suzanne Collins],[Locus Award Nominee for Best Young Adult...,First Edition,Hardcover,"[Teen & Young Adult, Literature & Fiction]",amazon_id_947,...,6.81,2008-09-14,Scholastic Press,4.515,3229044.0,The Hunger Games #1,The Hunger Games,The Hunger Games,0.609522,"{'_id_rule': 'first_non_null', '_id_input..."
1,amazon_id_1104,"[goodreads_id_2, amazon_id_1104]","[goodreads, amazon]",[J.K. Rowling],[J.K. Rowling],[Bram Stoker Award for Works for Young Re...,US Edition,Paperback,"[Children's Books, Growing Up & Facts of ...",amazon_id_1104,...,6.66,2004-09-01,Scholastic Paperbacks,4.650,1292467.0,Harry Potter #5,Harry Potter and the Order of the Phoenix,Harry Potter and the Order of the Phoenix...,0.625935,"{'_id_rule': 'first_non_null', '_id_input..."
2,amazon_id_352,"[goodreads_id_3, amazon_id_352]","[goodreads, amazon]",[Harper Lee],[Harper Lee],"[Pulitzer Prize for Fiction (1961), Audie...",None,Paperback,"[Literature & Fiction, Genre Fiction]",amazon_id_352,...,18.00,2006-05-06,Harper Perennial Modern Classics,4.490,2315936.0,To Kill a Mockingbird,To Kill a Mockingbird,To Kill a Mockingbird,0.591769,"{'_id_rule': 'first_non_null', '_id_input..."


In [39]:
# Our own consistency / density tables (same definitions as the lecture) for the report
cons_strict = consistency(list(dfs.values()), gt_used, FUSE_ATTRS, thr=None)
cons_tol = consistency(list(dfs.values()), gt_used, FUSE_ATTRS, thr=0.8)
print("=== Consistency of the input groups (share of groups without conflicting values) ===")
display(pd.DataFrame({"identical values": cons_strict, "similar values (>= 0.8)": cons_tol}).round(3))

dens_fused = density(fused_baseline, FUSE_ATTRS)
largest = max(dfs, key=lambda n: len(dfs[n]))
print(f"=== Density: largest input dataset ({largest}) vs fused ===")
display(pd.DataFrame({f"{largest} (input)": dens[largest].reindex(dens_fused.index), "fused": dens_fused}).round(3))

=== Consistency of the input groups (share of groups without conflicting values) ===


,identical values,similar values (>= 0.8)
title,0.412,0.430
publisher,0.439,0.580
publication_date,0.418,0.671
format,0.906,0.906
language,0.940,0.940
authors_author,0.747,0.782
page_count,0.418,0.536
rating,0.660,0.660
ratings_count,0.659,0.659
price,0.750,0.750


=== Density: largest input dataset (dbpedia) vs fused ===


,dbpedia (input),fused
title,1.000,1.000
publisher,0.795,0.890
publication_date,0.802,0.883
format,0.000,0.607
language,0.796,0.900
authors_author,0.909,0.964
page_count,0.616,0.754
rating,0.000,0.642
ratings_count,0.000,0.642
price,0.000,0.511


In [40]:
if df_eval is not None:
    fb = attach_eval_id(fused_baseline, expected_ids)
    evaluator = DataFusionEvaluator(baseline_strategy, debug=True, debug_file=str(OUT_DIR / "fusion_evaluation_debug_baseline.jsonl"),
                                    fusion_debug_logs=str(OUT_DIR / "fusion_debug_baseline.jsonl"))
    m = evaluator.evaluate(fused_df=fb, fused_id_column="eval_id", expected_df=df_eval_dev, expected_id_column="id")
    print("PyDI evaluator, overall accuracy (dev):", round(m["overall_accuracy"], 4))
    print("\n=== Baseline accuracy per attribute (dev), own computation ===")
    display(attribute_accuracy(fb, df_eval_dev, [a for a in EVAL_ATTRS if a in df_eval_dev.columns]).round(3))

[INFO] Fusion evaluation debug logging enabled; refer to data/output/data_fusion/fusion_evaluation_debug_baseline.jsonl for mismatch details.
[INFO] Starting fusion evaluation
[INFO] Evaluation complete: 0.704 overall accuracy (100/142)
[INFO] Evaluation mismatches by attribute (debug): 42 total
[INFO] 	Attribute                        |  Errors | Percentage
[INFO] 	───────────────────────────────────────────────────────
[INFO] 	page_count                       |      13 |     30.95%%
[INFO] 	publication_date                 |      11 |     26.19%%
[INFO] 	publisher                        |       9 |     21.43%%
[INFO] 	authors_author                   |       6 |     14.29%%
[INFO] 	title                            |       3 |      7.14%%


PyDI evaluator, overall accuracy (dev): 0.7042

=== Baseline accuracy per attribute (dev), own computation ===


,values,accuracy strict,accuracy tolerant
attribute,,,
title,29.0,0.793,0.897
authors_author,29.0,0.793,0.828
publisher,29.0,0.414,0.690
publication_date,29.0,0.621,0.621
page_count,26.0,0.192,0.500
OVERALL,142.0,0.570,0.711


## 9. Experiments: which function works for which attribute?

**Source reliability.** For every dataset and attribute: how often is the dataset's own value correct (tolerant comparison, dev part of the evaluation set)?
This is the evidence for the trust scores / source preference (metadata-based conflict resolution).

In [41]:
if df_eval is not None:
    rows = []
    comp_all = components(*corr_list_used)
    members = defaultdict(list)
    for rid, g in comp_all.items():
        members[g].append(rid)
    for a in [a for a in EVAL_ATTRS if a in df_eval_dev.columns]:
        for n in NAMES:
            ok = tot = 0
            for eid, true in df_eval_dev.set_index("id")[a].items():
                if is_null(true):
                    continue
                rec = [i for i in members.get(comp_all.get(eid, eid), [eid]) if source_of[i] == n]
                if not rec:
                    continue
                val = dfs[n].set_index("id").at[rec[0], a]
                if is_null(val):
                    continue
                tot += 1
                ok += similar_values(val, true, a, 0.8)
            rows.append({"attribute": a, "dataset": n, "values": tot, "correct": ok / tot if tot else np.nan})
    reliability = pd.DataFrame(rows).pivot(index="attribute", columns="dataset", values="correct")[NAMES]
    print("=== Share of correct values per source (only where the source has a value) ===")
    display(reliability.round(2))

=== Share of correct values per source (only where the source has a value) ===


dataset,goodreads,amazon,dbpedia
attribute,,,
authors_author,0.92,0.86,0.81
page_count,0.59,0.27,0.50
publication_date,0.77,0.77,0.55
publisher,0.86,0.77,0.47
title,0.96,0.93,0.76


**Grid of conflict resolution functions per attribute** (each run fuses only that attribute; for the trust-aware functions `prefer_higher_trust` and our own `publisher_vote` all 6 source orders are tried).
Function names are looked up in `PyDI.fusion`; those that do not exist in your PyDI version are skipped (see the list printed above, and the PyDI fusion wiki).
Ties are resolved in favour of the function listed first (simpler first), like in the identity-resolution notebook.

In [42]:
CANDIDATES = {   # attribute -> function names, simpler first; several names = aliases
    "title":            [("longest_string",), ("shortest_string",), ("voting", "vote"), ("prefer_higher_trust",)],
    "authors_author":   [("union",), ("intersection",), ("intersection_k_sources",), ("prefer_higher_trust",)],
    "publisher":        [("longest_string",), ("voting", "vote"), ("prefer_higher_trust",), ("publisher_vote",)],   # last: own function
    "publication_date": [("voting", "vote"), ("earliest",), ("most_recent",), ("prefer_higher_trust",)],
    "page_count":       [("median", "median_value"), ("average", "mean"), ("maximum", "max_value"), ("minimum", "min_value"),
                         ("voting", "vote"), ("prefer_higher_trust",)],
    "format":           [("longest_string",), ("voting", "vote"), ("prefer_higher_trust",)],
}
ORDERS = list(itertools.permutations(NAMES))
TRUST_AWARE = {prefer_higher_trust, publisher_vote}   # functions that take a per-attribute trust_map -> all 6 source orders are tried


def run_single(attr, f, order):
    """Fuse only `attr`. Datasets are always passed in the same order (NAMES), so tie-breaks of
    voting / longest_string etc. are identical in the grid and in the final strategy.
    Trust-aware functions (TRUST_AWARE) get the trust order as a per-attribute trust_map (overrides df.attrs['trust'])."""
    lvl = logging.getLogger().level
    logging.getLogger().setLevel(logging.WARNING)
    try:
        s = DataFusionStrategy(f"{attr}:{getattr(f, '__name__', 'f')}")
        if f in TRUST_AWARE:
            s.add_attribute_fuser(attr, f, trust_map=trust_from_order(order))
        else:
            s.add_attribute_fuser(attr, f)
        out = DataFusionEngine(s).run(datasets=[dfs[n] for n in NAMES], correspondences=corr_list_used, id_column="id")
    finally:
        logging.getLogger().setLevel(lvl)
    return attach_eval_id(out, expected_ids)


grid_rows = []
if df_eval is not None:
    for attr, options in CANDIDATES.items():
        if attr not in df_eval_dev.columns:
            continue
        for names in options:
            f = fn(*names)
            if f is None:
                print(f"skipped (not in PyDI.fusion): {names}")
                continue
            for order in (ORDERS if f in TRUST_AWARE else [tuple(NAMES)]):
                try:
                    acc = attribute_accuracy(run_single(attr, f, order), df_eval_dev, [attr]).loc[attr]
                except Exception as e:                      # keep going, show the problem
                    print(f"failed: {attr} / {names[0]} / {order}: {e!r}")
                    continue
                grid_rows.append({"attribute": attr, "function": names[0], "pos": options.index(names),
                                  "order": order if f in TRUST_AWARE else None,
                                  "trust order": " > ".join(order) if f in TRUST_AWARE else "-",
                                  "strict": acc["accuracy strict"], "tolerant": acc["accuracy tolerant"]})
    grid = pd.DataFrame(grid_rows)
    grid.drop(columns=["pos", "order"]).to_csv(OUT_DIR / "function_grid_dev.csv", index=False)
    for a in grid["attribute"].unique():
        print(f"\n=== {a}: accuracy on the dev part of the evaluation set ===")
        display(grid[grid["attribute"] == a].drop(columns=["attribute", "pos", "order"]).sort_values("tolerant", ascending=False).round(3).reset_index(drop=True))


=== title: accuracy on the dev part of the evaluation set ===


,function,trust order,strict,tolerant
0,voting,-,0.931,0.966
1,prefer_higher_trust,goodreads > amazon > dbpedia,0.931,0.966
2,prefer_higher_trust,goodreads > dbpedia > amazon,0.897,0.931
3,prefer_higher_trust,amazon > goodreads > dbpedia,0.897,0.931
4,longest_string,-,0.793,0.897
5,shortest_string,-,0.862,0.862
6,prefer_higher_trust,amazon > dbpedia > goodreads,0.759,0.828
7,prefer_higher_trust,dbpedia > goodreads > amazon,0.759,0.828
8,prefer_higher_trust,dbpedia > amazon > goodreads,0.724,0.793



=== authors_author: accuracy on the dev part of the evaluation set ===


,function,trust order,strict,tolerant
0,prefer_higher_trust,goodreads > dbpedia > amazon,0.828,0.897
1,prefer_higher_trust,amazon > goodreads > dbpedia,0.862,0.897
2,prefer_higher_trust,goodreads > amazon > dbpedia,0.793,0.862
3,prefer_higher_trust,dbpedia > amazon > goodreads,0.862,0.862
4,union,-,0.793,0.828
5,prefer_higher_trust,amazon > dbpedia > goodreads,0.828,0.828
6,prefer_higher_trust,dbpedia > goodreads > amazon,0.793,0.828
7,intersection,-,0.759,0.759
8,intersection_k_sources,-,0.759,0.759



=== publisher: accuracy on the dev part of the evaluation set ===


,function,trust order,strict,tolerant
0,prefer_higher_trust,amazon > goodreads > dbpedia,0.586,0.793
1,publisher_vote,amazon > goodreads > dbpedia,0.517,0.793
2,voting,-,0.586,0.759
3,prefer_higher_trust,goodreads > amazon > dbpedia,0.586,0.759
4,prefer_higher_trust,goodreads > dbpedia > amazon,0.586,0.759
5,publisher_vote,goodreads > amazon > dbpedia,0.483,0.759
6,publisher_vote,goodreads > dbpedia > amazon,0.483,0.759
7,longest_string,-,0.414,0.690
8,prefer_higher_trust,amazon > dbpedia > goodreads,0.379,0.621
9,prefer_higher_trust,dbpedia > amazon > goodreads,0.379,0.621



=== publication_date: accuracy on the dev part of the evaluation set ===


,function,trust order,strict,tolerant
0,voting,-,0.724,0.724
1,most_recent,-,0.724,0.724
2,prefer_higher_trust,goodreads > amazon > dbpedia,0.724,0.724
3,prefer_higher_trust,goodreads > dbpedia > amazon,0.724,0.724
4,prefer_higher_trust,amazon > goodreads > dbpedia,0.724,0.724
5,earliest,-,0.621,0.621
6,prefer_higher_trust,amazon > dbpedia > goodreads,0.621,0.621
7,prefer_higher_trust,dbpedia > goodreads > amazon,0.621,0.621
8,prefer_higher_trust,dbpedia > amazon > goodreads,0.621,0.621



=== page_count: accuracy on the dev part of the evaluation set ===


,function,trust order,strict,tolerant
0,prefer_higher_trust,goodreads > dbpedia > amazon,0.308,0.577
1,minimum,-,0.308,0.538
2,voting,-,0.269,0.538
3,prefer_higher_trust,goodreads > amazon > dbpedia,0.269,0.538
4,median,-,0.192,0.500
5,average,-,0.154,0.500
6,prefer_higher_trust,dbpedia > goodreads > amazon,0.308,0.500
7,prefer_higher_trust,amazon > goodreads > dbpedia,0.231,0.462
8,maximum,-,0.231,0.423
9,prefer_higher_trust,amazon > dbpedia > goodreads,0.269,0.423


**Provenance vs. no provenance** (exercise slide 10). For every attribute: the best function that only looks at the values
(`voting`, `longest_string`, `union`, `median`, ...) against the best function that also uses the provenance of the values, i.e. which
dataset a value comes from (`prefer_higher_trust`, our own `publisher_vote`, with the per-attribute trust order). Chosen on the dev part
(tolerant accuracy, ties -> simpler function), then both are also evaluated on the test part, which was not used for the choice.


In [24]:
if grid_rows:
    g = pd.DataFrame(grid_rows)
    g["provenance"] = g["order"].map(lambda o: o is not None)
    rows = []
    for a, sub in g.groupby("attribute", sort=False):
        row = {"attribute": a}
        for prov, label in [(False, "no provenance"), (True, "provenance")]:
            part = sub[sub["provenance"] == prov]
            if part.empty:
                continue
            b = part.sort_values(["tolerant", "pos"], ascending=[False, True], kind="stable").iloc[0]
            order = b["order"] if prov else tuple(NAMES)
            test = attribute_accuracy(run_single(a, fn(b["function"]), order), df_eval_test, [a]).loc[a]
            row.update({(label, "function"): b["function"] + (f" ({' > '.join(order)})" if prov else ""),
                        (label, "dev"): b["tolerant"], (label, "test"): test["accuracy tolerant"]})
        rows.append(row)
    prov_cmp = pd.DataFrame(rows).set_index("attribute")
    prov_cmp.columns = pd.MultiIndex.from_tuples(prov_cmp.columns)
    prov_cmp[("difference", "test")] = prov_cmp[("provenance", "test")] - prov_cmp[("no provenance", "test")]
    prov_cmp.to_csv(OUT_DIR / "provenance_comparison.csv")
    pd.set_option("display.max_colwidth", 70)
    print("=== Best function without vs. with provenance (tolerant accuracy; use this table in the report) ===")
    display(prov_cmp.round(3))


=== Best function without vs. with provenance (tolerant accuracy; use this table in the report) ===


no provenance                \
                      function    dev   test   
attribute                                      
title                   voting  0.966  0.933   
authors_author           union  0.828  0.933   
publisher               voting  0.759  0.633   
publication_date        voting  0.724  0.733   
page_count             minimum  0.538  0.621   

                                                          provenance         \
                                                            function    dev   
attribute                                                                     
title             prefer_higher_trust (goodreads > amazon > dbpedia)  0.966   
authors_author    prefer_higher_trust (goodreads > dbpedia > amazon)  0.897   
publisher         prefer_higher_trust (amazon > goodreads > dbpedia)  0.793   
publication_date  prefer_higher_trust (goodreads > amazon > dbpedia)  0.724   
page_count        prefer_higher_trust (goodreads > dbpedia > amazon)  0.577   

                        difference  
                   test       test  
attribute                           
title             0.900     -0.033  
authors_author    0.933      0.000  
publisher         0.633      0.000  
publication_date  0.733      0.000  
page_count        0.586     -0.034

## 10. Final strategy

For every evaluated attribute the best row of the grid is selected automatically (tolerant dev accuracy; ties -> the function listed first in
`CANDIDATES`, i.e. the simpler one). If that row is `prefer_higher_trust`, its **own source order** is used for this attribute
(per-attribute `trust_map`), because the sources are reliable for different attributes (see the reliability table: goodreads is best for
title, amazon for authors, goodreads/dbpedia for page_count). You can override the choice in `OVERRIDE`.

Attributes without ground truth are fixed by reasoning (`FIXED`, trust order `DEFAULT_ORDER`): ratings are subjective and change over
time -> median; price and language come from one source in most groups -> `prefer_higher_trust`. **TODO (team):** justify these choices in the report.

**Be careful with lenient evaluation:** the threshold 0.8 of the tolerant comparison counts e.g. "Dune" vs "Dune (novel)" as correct. Write down which cases this accepts and why that is fine for your use case.


In [43]:
DEFAULT_ORDER = ("amazon", "goodreads", "dbpedia")    # trust order for attributes WITHOUT ground truth (price, language, format)
OVERRIDE = {}                                           # e.g. {"page_count": "prefer_higher_trust"} -> best order of that function is used
FIXED = {"rating": ("median",), "ratings_count": ("median",), "price": ("prefer_higher_trust",),
         "language": ("prefer_higher_trust",), "format": ("prefer_higher_trust",)}

chosen = {}                                             # attribute -> (function name, trust order or None)
if grid_rows:
    g = pd.DataFrame(grid_rows)
    for a, sub in g.groupby("attribute"):
        if a in OVERRIDE:
            sub = sub[sub["function"] == OVERRIDE[a]]
        best = sub.sort_values(["tolerant", "pos"], ascending=[False, True], kind="stable").iloc[0]
        chosen[a] = (best["function"], best["order"])

final_strategy = DataFusionStrategy("Version_Final")
rows = []
for a in FUSE_ATTRS:
    name, order = chosen.get(a, (FIXED.get(a, ("prefer_higher_trust",))[0], None))
    f = fn(name) or prefer_higher_trust
    if f in TRUST_AWARE:
        order = order or DEFAULT_ORDER
        final_strategy.add_attribute_fuser(a, f, trust_map=trust_from_order(order))
    else:
        order = None
        final_strategy.add_attribute_fuser(a, f)
    rows.append({"attribute": a, "function": getattr(f, "__name__", name),
                 "trust order": " > ".join(order) if order else "-", "chosen by": "dev grid" if a in chosen else "fixed"})
add_eval_functions(final_strategy)
print("=== Final strategy (use this table in the report) ===")
display(pd.DataFrame(rows).set_index("attribute"))

final_engine = DataFusionEngine(final_strategy, debug=True, debug_file=str(OUT_DIR / "fusion_debug_final.jsonl"))
fused_final = final_engine.run(datasets=[dfs[n] for n in NAMES], correspondences=corr_list_used,
                               id_column="id", include_singletons=True)
fused_final.to_csv(OUT_DIR / "fused_books.csv", index=False)
print(len(fused_final), "records in the integrated dataset ->", OUT_DIR / "fused_books.csv")


[INFO] Registered fuser for attribute 'title' using rule 'voting'
[INFO] Registered fuser for attribute 'publisher' using rule 'prefer_higher_trust'
[INFO] Registered fuser for attribute 'publication_date' using rule 'voting'
[INFO] Registered fuser for attribute 'format' using rule 'prefer_higher_trust'
[INFO] Registered fuser for attribute 'language' using rule 'prefer_higher_trust'
[INFO] Registered fuser for attribute 'authors_author' using rule 'prefer_higher_trust'
[INFO] Registered fuser for attribute 'page_count' using rule 'prefer_higher_trust'
[INFO] Registered fuser for attribute 'rating' using rule 'median'
[INFO] Registered fuser for attribute 'ratings_count' using rule 'median'
[INFO] Registered fuser for attribute 'price' using rule 'prefer_higher_trust'
[INFO] Registered evaluation function for attribute 'title' with params {'threshold': 0.8}
[INFO] Registered evaluation function for attribute 'authors_author' with params {'threshold': 0.8}
[INFO] Registered evaluation 

=== Final strategy (use this table in the report) ===


,function,trust order,chosen by
attribute,,,
title,voting,-,dev grid
publisher,prefer_higher_trust,amazon > goodreads > dbpedia,dev grid
publication_date,voting,-,dev grid
format,prefer_higher_trust,amazon > goodreads > dbpedia,fixed
language,prefer_higher_trust,amazon > goodreads > dbpedia,fixed
authors_author,prefer_higher_trust,goodreads > dbpedia > amazon,dev grid
page_count,prefer_higher_trust,goodreads > dbpedia > amazon,dev grid
rating,median,-,fixed
ratings_count,median,-,fixed


[INFO] Fusion debug logging enabled; refer to data/output/data_fusion/fusion_debug_final.jsonl for detailed traces.
[INFO] Starting data fusion with strategy 'Version_Final'
[INFO] *    Loading correspondences    *
[INFO] Correspondence ID coverage: matched 27135 of 27135 unique IDs
[INFO] Created 137313 record groups from 14900 correspondences
[INFO] Group Size Distribution of 137313 clusters:
[INFO] 	Cluster Size	| Frequency	| Percentage
[INFO] 	──────────────────────────────────────────────────
[INFO] 		2	|	10965	|	7.99%
[INFO] 		3	|	1735	|	1.26%
[INFO] Attribute Consistencies:
[INFO]     _id: 0.00
[INFO]     authors_author: 0.74
[INFO]     authors_raw: 0.41
[INFO]     awards_award: 1.00
[INFO]     edition: 0.93
[INFO]     format: 0.90
[INFO]     genres_genre: 0.21
[INFO]     id: 0.00
[INFO]     language: 0.94
[INFO]     page_count: 0.40
[INFO]     price: 0.72
[INFO]     publication_date: 0.41
[INFO]     publisher: 0.41
[INFO]     rating: 0.66
[INFO]     ratings_count: 0.66
[INFO]  

137313 records in the integrated dataset -> data/output/data_fusion/fused_books.csv


In [44]:
if df_eval is not None:
    ff = attach_eval_id(fused_final, expected_ids)
    evaluator = DataFusionEvaluator(final_strategy, debug=True, debug_file=str(OUT_DIR / "fusion_evaluation_debug_final.jsonl"),
                                    fusion_debug_logs=str(OUT_DIR / "fusion_debug_final.jsonl"))
    attrs_eval = [a for a in EVAL_ATTRS if a in df_eval.columns]
    final_metrics = evaluator.evaluate(fused_df=ff, fused_id_column="eval_id", expected_df=df_eval_test, expected_id_column="id")
    print("PyDI evaluator, overall accuracy on the TEST part:", round(final_metrics["overall_accuracy"], 4))
    cmp = pd.concat({"baseline (test)": attribute_accuracy(attach_eval_id(fused_baseline, expected_ids), df_eval_test, attrs_eval),
                     "final (test)": attribute_accuracy(ff, df_eval_test, attrs_eval)}, axis=1)
    print("\n=== Accuracy per attribute on the TEST part (use this table in the report) ===")
    display(cmp.round(3))

[INFO] Fusion evaluation debug logging enabled; refer to data/output/data_fusion/fusion_evaluation_debug_final.jsonl for mismatch details.
[INFO] Starting fusion evaluation
[INFO] Evaluation complete: 0.765 overall accuracy (114/149)
[INFO] Evaluation mismatches by attribute (debug): 35 total
[INFO] 	Attribute                        |  Errors | Percentage
[INFO] 	───────────────────────────────────────────────────────
[INFO] 	page_count                       |      12 |     34.29%%
[INFO] 	publisher                        |      11 |     31.43%%
[INFO] 	publication_date                 |       8 |     22.86%%
[INFO] 	title                            |       2 |      5.71%%
[INFO] 	authors_author                   |       2 |      5.71%%


PyDI evaluator, overall accuracy on the TEST part: 0.7651

=== Accuracy per attribute on the TEST part (use this table in the report) ===


baseline (test)                                    \
                          values accuracy strict accuracy tolerant   
attribute                                                            
title                       30.0           0.833             0.833   
authors_author              30.0           0.933             0.933   
publisher                   30.0           0.267             0.567   
publication_date            30.0           0.700             0.700   
page_count                  29.0           0.241             0.552   
OVERALL                    149.0           0.597             0.718   

                 final (test)                                    
                       values accuracy strict accuracy tolerant  
attribute                                                        
title                    30.0           0.933             0.933  
authors_author           30.0           0.933             0.933  
publisher                30.0           0.433             0.633  
publication_date         30.0           0.733             0.733  
page_count               29.0           0.414             0.586  
OVERALL                 149.0           0.691             0.765

## 11. Error analysis

Every value that is still wrong after fusion (tolerant comparison, **whole** evaluation set = dev + test), with the fused value, the true value
and the value of each source. Each error gets one cause (checked in this order):

1. **suspected identity-resolution error**: the cleaned titles of the records in the group hardly overlap (< 50 % of the words of the shorter title), so the records probably describe different books
2. **no value in any source**: nothing to fuse (density problem, not a fusion problem)
3. **wrong value chosen**: the true value is in at least one source, but the conflict resolution function picked another one -> only these can be fixed by a better function
4. **true value in no source**: all sources disagree with the truth (other edition / reprint in the sources, dbpedia describes the work rather than the edition, imprint vs. parent publisher, ...)

The full table is saved to `fusion_errors.csv` for the report.


In [45]:
def title_overlap(titles):
    """Smallest word overlap (relative to the shorter title) between any two titles of a group."""
    toks = [tokens(t) for t in titles if not is_null(t)]
    pairs = [(a, b) for k, a in enumerate(toks) for b in toks[k + 1:] if a and b]
    return min((len(a & b) / min(len(a), len(b)) for a, b in pairs), default=1.0)


CAUSES = ["suspected identity-resolution error", "no value in any source", "wrong value chosen", "true value in no source"]

if df_eval is not None:
    ex = df_eval.set_index("id")
    f = ff[ff["eval_id"].notna()].drop_duplicates("eval_id").set_index("eval_id")
    comp_all = components(*corr_list_used)
    members = defaultdict(list)
    for rid, g in comp_all.items():
        members[g].append(rid)
    by_src = {n: d.set_index("id") for n, d in dfs.items()}
    rows = []
    for eid in ex.index:
        recs = members.get(comp_all.get(eid, eid), [eid])
        overlap = title_overlap([by_src[source_of[i]].at[i, "title"] for i in recs])
        for a in attrs_eval:
            true, got = ex.at[eid, a], f.at[eid, a] if eid in f.index else None
            if is_null(true) or (not is_null(got) and similar_values(got, true, a, 0.8)):
                continue
            src = {source_of[i]: by_src[source_of[i]].at[i, a] for i in recs}
            in_source = any((not is_null(v)) and similar_values(v, true, a, 0.8) for v in src.values())
            if overlap < 0.5:
                cause = CAUSES[0]
            elif all(is_null(v) for v in src.values()):
                cause = CAUSES[1]
            elif in_source:
                cause = CAUSES[2]
            else:
                cause = CAUSES[3]
            rows.append({"entity": eid, "attribute": a, "cause": cause, "true": true, "fused": got,
                         **{n: src.get(n) for n in NAMES}, "split": "dev" if eid in eval_dev_ids else "test"})
    errors = pd.DataFrame(rows)
    errors.to_csv(OUT_DIR / "fusion_errors.csv", index=False)
    n_vals = int(sum((~ex[a].map(is_null)).sum() for a in attrs_eval))
    print(f"{len(errors)} of {n_vals} evaluated values are wrong or missing -> {OUT_DIR / 'fusion_errors.csv'}")
    if len(errors):
        summary = pd.crosstab(errors["attribute"], pd.Categorical(errors["cause"], categories=CAUSES), margins=True, margins_name="total",
                              dropna=False, colnames=["cause"])
        print("\n=== Errors per attribute and cause (use this table in the report) ===")
        display(summary)
        pd.set_option("display.max_colwidth", 40)
        pd.set_option("display.max_rows", 200)
        display(errors.sort_values(["cause", "attribute", "entity"], key=lambda c: c.map(CAUSES.index) if c.name == "cause" else c)
                      .reset_index(drop=True))


64 of 291 evaluated values are wrong or missing -> data/output/data_fusion/fusion_errors.csv

=== Errors per attribute and cause (use this table in the report) ===


cause,suspected identity-resolution error,no value in any source,wrong value chosen,true value in no source,total
attribute,,,,,
authors_author,2,0,3,0,5
page_count,3,0,3,17,23
publication_date,3,1,1,11,16
publisher,1,0,7,9,17
title,0,0,3,0,3
total,9,1,17,37,64


,entity,attribute,cause,true,fused,goodreads,amazon,dbpedia,split
0,amazon_id_11992,authors_author,suspected identity-resolution error,"[Vince Mills, Kyle Flynn]",[Errol Flynn],None,[Vince Flynn],[Errol Flynn],dev
1,dbpedia_id_3299,authors_author,suspected identity-resolution error,[David French],[August Strindberg],[August Strindberg],None,[August Strindberg],dev
2,dbpedia_id_29386,page_count,suspected identity-resolution error,334.0,274.0,274.0,None,252.0,test
3,dbpedia_id_47159,page_count,suspected identity-resolution error,272.0,264.0,264.0,None,263.0,dev
4,dbpedia_id_52098,page_count,suspected identity-resolution error,250.0,256.0,256.0,None,259.0,test
5,amazon_id_11992,publication_date,suspected identity-resolution error,2020,1937-01-01,None,NaN,1937-01-01,dev
6,dbpedia_id_29386,publication_date,suspected identity-resolution error,2013,2000-12-01,2000-12-01,None,NaN,test
7,dbpedia_id_47159,publication_date,suspected identity-resolution error,2016,2009-02-20,2009-02-20,None,2009-01-01,dev
8,amazon_id_11992,publisher,suspected identity-resolution error,Simon & Schuster,None,None,NaN,NaN,dev
9,dbpedia_id_27788,publication_date,no value in any source,1994,None,NaN,None,NaN,test


## 12. Summary for the report

The three final questions of the exercise (largest input dataset as reference):
1. How many additional entities compared to the largest input dataset?
2. How much did the density increase compared to the largest input dataset?
3. What is the overall accuracy of the final dataset according to the evaluation set?

In [46]:
largest = max(dfs, key=lambda n: len(dfs[n]))
n_input = sum(len(d) for d in dfs.values())
n_fused, n_largest = len(fused_final), len(dfs[largest])
n_merged_away = int((fused_final["_fusion_sources"].map(len) - 1).sum())
d_in, d_out = density(dfs[largest], FUSE_ATTRS)["TABLE"], density(fused_final, FUSE_ATTRS)["TABLE"]

print(f"input records: {n_input:,} - {n_merged_away:,} merged away = {n_input - n_merged_away:,} "
      f"({'OK' if n_input - n_merged_away == n_fused else 'MISMATCH'}: integrated dataset has {n_fused:,} records)")
print(f"largest input dataset: {largest} with {n_largest:,} records\n")
print(f"1. entities in the integrated dataset: {n_fused:,}  ->  {n_fused - n_largest:,} additional entities (+{(n_fused - n_largest) / n_largest:.1%})")
print(f"2. table density: {d_in:.3f} -> {d_out:.3f}  (+{d_out - d_in:.3f})")
if df_eval is not None:
    acc = attribute_accuracy(ff, df_eval_test, attrs_eval).loc["OVERALL"]
    acc_all = attribute_accuracy(ff, df_eval, attrs_eval).loc["OVERALL"]
    print(f"3. overall accuracy on the TEST part ({len(df_eval_test)} entities, {int(acc['values'])} values): "
          f"strict {acc['accuracy strict']:.3f}, tolerant {acc['accuracy tolerant']:.3f}")
    print(f"   (for reference, whole evaluation set incl. the dev part used to choose the functions, {len(df_eval)} entities, "
          f"{int(acc_all['values'])} values: strict {acc_all['accuracy strict']:.3f}, tolerant {acc_all['accuracy tolerant']:.3f})")
else:
    print("3. overall accuracy: create the evaluation set first")

# The table density of the integrated dataset is lower than that of goodreads / amazon alone, because it also contains the
# unmatched dbpedia records (dbpedia has no format, rating, ratings_count, price). Split by record type for the report:
kind = fused_final["_fusion_source_datasets"].map(lambda s: "merged (>= 2 sources)" if len(s) > 1 else f"only in {s[0]}")
by_kind = pd.DataFrame({k: {"records": len(g), "density": density(g, FUSE_ATTRS)["TABLE"]} for k, g in fused_final.groupby(kind)}).T
by_kind.loc["integrated dataset (all)"] = [n_fused, d_out]
by_kind["records"] = by_kind["records"].astype(int)
print("\n=== Density of the integrated dataset by record type (use this table in the report) ===")
display(by_kind.round(3))
print("\n=== Density of the input datasets, for comparison ===")
display(pd.DataFrame({n: {"records": len(d), "density": density(d, FUSE_ATTRS)["TABLE"]} for n, d in dfs.items()}).T.round(3)
        .astype({"records": int}))


largest input dataset: dbpedia with 59,270 records
1. entities in the integrated dataset: 137,313  ->  78,043 additional entities (+131.7%)
2. table density: 0.492 -> 0.779  (+0.288)
3. overall accuracy on the test part of the evaluation set: strict 0.691, tolerant 0.765

fused records that merge >= 2 source records: 12700


### Report checklist (Data Fusion part, exercise slide 16)
* provenance data you added and why you set the values as you did (section 6)
* density of inputs and fused data (sections 4 and 8), consistency per attribute (section 8)
* size, content and creation of the evaluation set, external source and why you chose it (section 7)
* functions tried per attribute, own functions if any, accuracy table with the best function per attribute (sections 9, 10)
* analysis of the remaining errors, e.g. "the true value is in none of the datasets" (section 11)
* `FILTER_MODE` (section 5): no effect, because all 12,700 groups are already clean (at most one record per dataset, thanks to the
  cluster cleanup after identity resolution) -> "clean_only" gives the same result; state this in one sentence
* strict vs. tolerant evaluation: tolerant rules are publisher = normalised names, publication_date = same year, page_count = +-2 %,
  title / authors = word Jaccard >= 0.8; titles and author lists are cleaned before fusion (section 3)
* final three numbers (section 12), with the density-by-record-type table (merged records vs. unmatched dbpedia records)